# Use Case 3: Detecting "Please Get Me a Human" (the `Noul` primitive)

**Recap so far:**
- `Choice` → pick one label out of a list (notebook 1)
- `Score` → rate on a scale with described levels (notebook 2)
- `Noul` → today's one: answer a **yes/no** question, as a probability.

**The scenario (same helpdesk):** a customer is chatting with us. We want to automatically detect: *is this person asking to talk to a real human?* If yes, we should stop the bot and route them to a person.

### 🎚️ Analogy

Instead of baskets or mailboxes, imagine a single dial with "No" on the far left and "Yes" on the far right. The person reading the message doesn't pick a basket — they just push a needle somewhere between the two ends.

- Needle pinned hard right (near 1.0) → confident "yes."
- Needle pinned hard left (near 0.0) → confident "no."
- Needle sitting right in the middle (near 0.5) → "honestly, could go either way."

There's no separate confidence number here — the needle's position alone tells you both *which way* it leans and *how sure* it is, all at once.

**Example 1:** "Can I speak to someone? The bot keeps repeating itself." — the person reading it pushes the needle almost all the way to "Yes" → `noul = 0.95`.

**Example 2:** "What are your opening hours?" — nothing here asks for a person, so the needle stays pinned at "No" → `noul` near 0.02.

Keep this picture in mind — everything below is just how we describe the question and the message to the computer, and how we read back where the needle landed.

### Why not just `if "human" in message:`?

Because people don't say it that way. "Can I speak to someone?", "Is there an actual person I can talk to?", "This bot isn't helping, get me support" — none of those contain the word "human", but they all mean the same thing. `Noul` reads the *meaning*, not the exact words — same intelligence idea as notebook 1, just producing a yes/no instead of a category.

## Step 1 — Install the SDK

In [ ]:
%pip install typesafe-sdk

## Step 2 — Add your API key

In [ ]:
import os

# 👉 PASTE YOUR TYPESAFE API KEY BELOW, between the quotes
os.environ["TYPESAFE_API_KEY"] = "PASTE_YOUR_API_KEY_HERE"

print("Key saved for this session.")

## Step 3 — Create the client

Same as always, we just import `Noul` this time instead of `Choice`/`Score`.

In [ ]:
from typesafe_sdk import Noul, TypeSafeClient

client = TypeSafeClient()
print("Client is ready.")

## Step 4 — The "state" (the customer's message)

In [ ]:
customer_message = "Can I speak to someone? I've asked the bot three times and it keeps repeating itself."

state = {"document": customer_message}

## Step 5 — Define the question(s)

A `Noul` question just needs `instructions` — the yes/no question in plain English. No `criteria` needed for simple cases (you can add it later if the yes/no line is ever blurry).

**Best practice (straight from the docs):** ask **one condition per `Noul`**. If you need to check several things at once, add several `Noul` questions to the same dictionary — like a checklist. Here we ask two things about the same message:

1. Is the customer asking for a human?
2. Does the customer sound frustrated?

Both get answered in a single request.

In [ ]:
questions = {
    "wants_human": Noul(
        instructions="Is the customer asking to speak to a human agent?",
    ),
    "is_frustrated": Noul(
        instructions="Does the customer sound frustrated or annoyed?",
    ),
}

## Step 6 — Send it to TypeSafe

Same method as always: `client.system_one(state=..., questions=...)`.

In [ ]:
response = client.system_one(
    state=state,
    questions=questions,
)

response

## Step 7 — Read the answer

For `Noul` questions, use `response.nouls["your_question_name"]`. Unlike `Choice`/`Score`, there's just **one number**: `.noul`, from 0 to 1.

How to read that number:
- **Close to 1.0** → "yes", and it's confident about it.
- **Close to 0.0** → "no", and it's confident about it.
- **Close to 0.5** → genuinely unsure / ambiguous — a 50/50 coin flip.

There's no separate `confidence` field here — the number itself already tells you both "which way" (above or below 0.5) **and** "how sure" (how far from 0.5).

In [ ]:
wants_human = response.nouls["wants_human"]
is_frustrated = response.nouls["is_frustrated"]

print("wants_human  :", wants_human.noul)
print("is_frustrated:", is_frustrated.noul)

## Step 8 — Turn the number into a decision

In real code, you don't print the raw number — you compare it against a **threshold** to make a yes/no decision your program can act on. A common starting threshold is `0.8` ("pretty confident"), but you can raise or lower it depending on how costly a wrong guess is.

In [ ]:
THRESHOLD = 0.8

if wants_human.noul > THRESHOLD:
    print("→ Route this chat to a human agent.")
else:
    print("→ Let the bot keep handling this.")

## Step 9 — Try it yourself

Edit `customer_message_2` below. Try something that's clearly *not* asking for a human (e.g. "What are your opening hours?") and something borderline (e.g. "This isn't working out") and watch the number move.

In [ ]:
# ✏️ Edit this text, then re-run this cell (Shift+Enter)
customer_message_2 = "What are your opening hours on weekends?"

response_2 = client.system_one(
    state={"document": customer_message_2},
    questions=questions,  # reusing the same two questions from Step 5
)

print("wants_human  :", response_2.nouls["wants_human"].noul)
print("is_frustrated:", response_2.nouls["is_frustrated"].noul)

## Step 10 — Clean up

In [ ]:
client.close()
print("Closed.")

## Recap — what did we just learn?

1. `Noul(instructions=...)` asks a plain yes/no question and returns one number from 0 to 1 — the probability the answer is "yes".
2. Read it like a dial, not a label: near 1 = yes, near 0 = no, near 0.5 = unsure.
3. `response.nouls["your_question_name"].noul` gets the number.
4. Ask **one condition per `Noul`**, and just add more entries to the `questions` dict for a checklist — they all get answered in the same request.
5. In real code, you compare the number to a **threshold** (e.g. `> 0.8`) to turn it into an actual `if/else` decision.

You've now seen all 3 basic TypeSafe question types:
- **Choice** — pick one of several labels
- **Score** — rate on a described scale
- **Noul** — yes/no as a probability

Every real use case from the TypeSafe use-case map (guardrails, fraud detection, recruiting, content moderation, RAG reranking, etc.) is just these 3 building blocks combined — usually several `Choice`/`Score`/`Noul` questions asked about the same `state` in one request.

👉 Tell me which real use case from the map you'd like to build next (e.g. LLM guardrails, RAG search reranking, moderation) and we'll turn it into the next notebook.